### **1: Install Required Libraries**

In [19]:
!pip -q install langchain langchain-community langchain-text-splitters langchain_nvidia_ai_endpoints langchain_core langchain_classic langchain-pinecone langchain-huggingface
!pip -q install bitsandbytes accelerate transformers
!pip -q install datasets loralib sentencepiece
!pip -q install pypdf
!pip -q install sentence_transformers
!pip -q install unstructured #The website text is unstructured
!pip install tokenizers
!pip install xformers
!pip install pinecone-client
!pip -q install huggingface_hub
!pip install huggingface-hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.8/125.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.8/571.8 kB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 587.6/587.6 kB 39.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.3/259.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.2 MB/s eta 0:00:00


### **2: Now import all libraries**

In [21]:
from langchain_community.document_loaders import UnstructuredURLLoader
from langchain_text_splitters import CharacterTextSplitter

# 1. FIXED NVIDIA IMPORT: Use the official partner package
from langchain_nvidia_ai_endpoints import ChatNVIDIA

# 2. FIXED PINECONE IMPORTS: Pinecone now uses its own partner package
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone

from langchain_classic.chains import RetrievalQAWithSourcesChain

# 3. FIXED HUGGING FACE IMPORTS: Moved to the huggingface partner package
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_huggingface import HuggingFacePipeline

from transformers import AutoTokenizer, AutoModelForCausalLM
from transformers import pipeline
from huggingface_hub import notebook_login
import textwrap
import sys
import os
import torch

In [22]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.


True

### **3: Pass the Urls and extract the data from urls**

In [23]:
URLs=[
    'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522',
    'https://www.mosaicml.com/blog/mpt-7b',
    'https://stability.ai/blog/stability-ai-launches-the-first-of-its-stablelm-suite-of-language-models',
    'https://lmsys.org/blog/2023-03-30-vicuna/'
]

In [24]:
loaders = UnstructuredURLLoader(urls=URLs)
data = loaders.load()

In [25]:
data

[Document(metadata={'source': 'https://blog.gopenai.com/paper-review-llama-2-open-foundation-and-fine-tuned-chat-models-23e539522'}, page_content='Please enable cookies.\n\nSorry, you have been blocked\n\nYou are unable to access medium.com\n\nWhy have I been blocked?\n\nThis website is using a security service to protect itself from online attacks. The action you just performed triggered the security solution. There are several actions that could trigger this block including submitting a certain word or phrase, a SQL command or malformed data.\n\nWhat can I do to resolve this?\n\nYou can email the site owner to let them know you were blocked. Please include what you were doing when this page came up and the Cloudflare Ray ID found at the bottom of this page.\n\nCloudflare Ray ID: a3a53c5a2a226387 • Your IP: 34.125.62.72 • Performance & security by Cloudflare'),
 Document(metadata={'source': 'https://www.mosaicml.com/blog/mpt-7b'}, page_content='Skip to main content\n\nAI Research\n\nM

In [26]:
len(data)

4

### **4: Split the data into Chunks**

In [28]:
text_splitter = CharacterTextSplitter(
    separator = '\n',
    chunk_size = 1000,
    chunk_overlap = 200
)

In [29]:
text_chunks = text_splitter.split_documents(data)

In [30]:
len(text_chunks)

61

### **5: Download the Huggingface embeddings**

In [31]:
embedding = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [32]:
query_result = embedding.embed_query("Hello World")
len(query_result)

384

In [34]:
#query_result

### **6: Conver the txt chunks into embedding to crete knowledge**

In [82]:
from getpass import getpass
from google.colab import userdata
import os

try:
    # Try to load securely from Colab Secrets if it's configured
    os.environ['PINECONE_API_KEY'] = userdata.get('PINECONE_API_KEY')
except Exception:
    # If not found in Secrets, prompt securely at runtime (hides typing)
    print("PINECONE_API_KEY not found in Colab Secrets.")
    os.environ['PINECONE_API_KEY'] = getpass("Please enter your Pinecone API Key: ")

os.environ['PINECONE_API_ENVIRON'] = 'Environ_key'
PINECONE_API_KEY = os.environ.get('PINECONE_API_KEY')
PINECONE_API_ENVIRON = os.environ.get('PINECONE_API_ENVIRON')

In [39]:
# Initialize the modern Pinecone client using the imported Pinecone class
pc = Pinecone(api_key=PINECONE_API_KEY)

In [41]:
index_name = 'llama1'

In [74]:
# Properly index documents including their metadata (so each chunk retains its 'source' URL)
vectorstores = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embedding,
    index_name=index_name
)

In [82]:
from huggingface_hub import notebook_login
notebook_login()


In [45]:
!hf auth login --token #API_ACCES_TOKEN

Hint: A new version of huggingface_hub (1.31.0) is available! You are using version 1.29.0.
To update, run: hf update
Hint: The `hf-cli` skill is not installed. Run `hf skills add -g --claude` to teach your AI agents how to use the `hf` CLI.
The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `hf`CLI if you want to set the git credential as well.
Token is valid (permission: read).
The token `llama2` has been saved to /root/.cache/huggingface/stored_tokens
Your token has been saved to /root/.cache/huggingface/token
Login successful.
The current active token is: `llama2`


### **8: Create LLM**

In [57]:
#model = "TheBloke/Llama-2-13B-GGML"

In [60]:
# Override to use an open, non-gated PyTorch Hugging Face version of Llama-2-13B
model = "NousResearch/Llama-2-7b-chat-hf"

tokenizer = AutoTokenizer.from_pretrained(model)

model = AutoModelForCausalLM.from_pretrained(
    model,
    device_map='auto',
    torch_dtype=torch.float16,
    token=True
)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/26.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [62]:
# Create the Hugging Face text-generation pipeline first
hf_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.01,
    repetition_penalty=1.1
)

# Wrap it into LangChain's HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=hf_pipeline)

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'temperature', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [63]:
llm.invoke("What is the capital of India?")

[transformers] Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


'What is the capital of India?\nWhat is the currency of India?\nWhat is the population of India?\nWhat is the official language of India?\nWhat is the climate of India?\nWhat are the major industries of India?\nWhat are the major agricultural products of India?\nWhat are the major minerals found in India?\nWhat are the major tourist destinations in India?\nWhat are the major festivals celebrated in India?\nWhat are the major sports played in India?\nWhat are the major educational institutions in India?\nWhat are the major research centers in India?\nWhat are the major cultural attractions in India?\nWhat are the major historical sites in India?\nWhat are the major religious places in India?\nWhat are the major transportation systems in India?\nWhat are the major cities in India?\nWhat are the major states in India?\nWhat are the major union territories in India?\nWhat are the major districts in India?\nWhat are the major municipal corporations in India?\nWhat are the major panchayats i

### **9: ReterivalIQA**

In [67]:
from langchain_classic.chains import RetrievalQAWithSourcesChain

In [68]:
query = "How good is Vicuna"

In [69]:
docs = vectorstores.similarity_search(query, k=3)

In [70]:
docs

[Document(id='e0d51bc0-9641-4245-8386-1f2a9096fad0', metadata={}, page_content='‹ Back to Blog\n‹ Back to Blog\nVicuna: An Open-Source Chatbot Impressing GPT-4 with 90%* ChatGPT Quality\nThe Vicuna TeamMarch 30, 2023\nWe introduce Vicuna-13B, an open-source chatbot trained by fine-tuning LLaMA on user-shared conversations collected from ShareGPT. Preliminary evaluation using GPT-4 as a judge shows Vicuna-13B achieves more than 90%* quality of OpenAI ChatGPT and Google Bard while outperforming other models like LLaMA and Stanford Alpaca in more than 90%* of cases. The cost of training Vicuna-13B is around $300. The code and weights, along with an online demo, are publicly available for non-commercial use.\nVicuna (generated by stable diffusion 2.1)\nAccording to a fun and non-scientific evaluation with GPT-4. Further rigorous evaluation is needed.\nHow Good is Vicuna?'),
 Document(id='f1065be6-7a91-4914-89d6-b6fa18cf07c9', metadata={}, page_content='Vicuna (generated by stable diffusion

In [77]:
from langchain_core.prompts import PromptTemplate

# Define a custom document prompt that falls back to 'Unknown' if 'source' is not in the metadata
fallback_doc_prompt = PromptTemplate(
    template="Content: {page_content}\nSource: {source}",
    input_variables=["page_content"],
    default_values={"source": "Unknown Source"}
)

# Create the chain pointing to the fallback document prompt template
chain = RetrievalQAWithSourcesChain.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstores.as_retriever(),
    chain_type_kwargs={"document_prompt": fallback_doc_prompt}
)

In [79]:
# Fetch documents from the retriever
query = "How good is Vicuna"
retrieved_docs = vectorstores.as_retriever().invoke(query)

# Dynamically inject the missing metadata source so it doesn't crash
for doc in retrieved_docs:
    if 'source' not in doc.metadata or not doc.metadata['source']:
        doc.metadata['source'] = "Unknown Source"

# Run the combine documents chain manually with the clean documents to bypass validation errors
response = chain.combine_documents_chain.invoke({
    "input_documents": retrieved_docs,
    "question": query
})

print(response['output_text'])